In [2]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("sonardataset.csv")

# Assign column names
df.columns = [f'V{i}' for i in range(1,61)] + ['Target']

print(df.shape)
print(df.head())
print(df.info())

(208, 61)
       V1      V2      V3      V4      V5      V6      V7      V8      V9  \
0  0.0200  0.0371  0.0428  0.0207  0.0954  0.0986  0.1539  0.1601  0.3109   
1  0.0453  0.0523  0.0843  0.0689  0.1183  0.2583  0.2156  0.3481  0.3337   
2  0.0262  0.0582  0.1099  0.1083  0.0974  0.2280  0.2431  0.3771  0.5598   
3  0.0100  0.0171  0.0623  0.0205  0.0205  0.0368  0.1098  0.1276  0.0598   
4  0.0762  0.0666  0.0481  0.0394  0.0590  0.0649  0.1209  0.2467  0.3564   

      V10  ...     V52     V53     V54     V55     V56     V57     V58  \
0  0.2111  ...  0.0027  0.0065  0.0159  0.0072  0.0167  0.0180  0.0084   
1  0.2872  ...  0.0084  0.0089  0.0048  0.0094  0.0191  0.0140  0.0049   
2  0.6194  ...  0.0232  0.0166  0.0095  0.0180  0.0244  0.0316  0.0164   
3  0.1264  ...  0.0121  0.0036  0.0150  0.0085  0.0073  0.0050  0.0044   
4  0.4459  ...  0.0031  0.0054  0.0105  0.0110  0.0015  0.0072  0.0048   

      V59     V60  Target  
0  0.0090  0.0032       R  
1  0.0052  0.0044       R 

In [3]:
print(df.isnull().sum())

V1        0
V2        0
V3        0
V4        0
V5        0
         ..
V57       0
V58       0
V59       0
V60       0
Target    0
Length: 61, dtype: int64


In [4]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['Target'] = le.fit_transform(df['Target'])  # M=1, R=0

In [5]:
from sklearn.preprocessing import StandardScaler

X = df.drop('Target', axis=1)
y = df['Target']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

In [7]:
from sklearn.neural_network import MLPClassifier

model = MLPClassifier(hidden_layer_sizes=(32,), max_iter=500)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

In [8]:
y_pred = (model.predict(X_test) > 0.5).astype(int)

In [9]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))

Accuracy: 0.9523809523809523
Precision: 0.8888888888888888
Recall: 1.0
F1 Score: 0.9411764705882353


In [10]:
#Hyperparameter Tuning
from sklearn.neural_network import MLPClassifier

model = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation='relu',
    solver='adam',
    max_iter=500,
    random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

In [11]:
base_model = MLPClassifier(hidden_layer_sizes=(32,), max_iter=500, random_state=42)
base_model.fit(X_train, y_train)

y_pred_base = base_model.predict(X_test)

print("=== Base Model ===")
print("Accuracy:", accuracy_score(y_test, y_pred_base))
print("Precision:", precision_score(y_test, y_pred_base))
print("Recall:", recall_score(y_test, y_pred_base))
print("F1 Score:", f1_score(y_test, y_pred_base))

=== Base Model ===
Accuracy: 0.9047619047619048
Precision: 0.8
Recall: 1.0
F1 Score: 0.888888888888889


In [12]:
tuned_model = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation='relu',
    solver='adam',
    max_iter=500,
    random_state=42
)

tuned_model.fit(X_train, y_train)

y_pred_tuned = tuned_model.predict(X_test)

print("\n=== Tuned Model ===")
print("Accuracy:", accuracy_score(y_test, y_pred_tuned))
print("Precision:", precision_score(y_test, y_pred_tuned))
print("Recall:", recall_score(y_test, y_pred_tuned))
print("F1 Score:", f1_score(y_test, y_pred_tuned))


=== Tuned Model ===
Accuracy: 0.9047619047619048
Precision: 0.8333333333333334
Recall: 0.9375
F1 Score: 0.8823529411764706


In [13]:
comparison = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score"],
    "Base Model": [
        accuracy_score(y_test, y_pred_base),
        precision_score(y_test, y_pred_base),
        recall_score(y_test, y_pred_base),
        f1_score(y_test, y_pred_base)
    ],
    "Tuned Model": [
        accuracy_score(y_test, y_pred_tuned),
        precision_score(y_test, y_pred_tuned),
        recall_score(y_test, y_pred_tuned),
        f1_score(y_test, y_pred_tuned)
    ]
})

print(comparison)

      Metric  Base Model  Tuned Model
0   Accuracy    0.904762     0.904762
1  Precision    0.800000     0.833333
2     Recall    1.000000     0.937500
3   F1 Score    0.888889     0.882353
